In [ ]:
!pip install langchain langchain-core langsmith -q

In [ ]:
import os

# Enable tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"

LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

if not LANGCHAIN_API_KEY:
    print("warning: LangSmith API key not found. Tracing may not work.")

In [ ]:
#Imports
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
#Prompt Templates
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract:
- Skills
- Experience
- Tools

Resume:
{resume}
"""
)

match_prompt = PromptTemplate(
    input_variables=["resume", "job_desc"],
    template="""
Compare resume with job description.

Resume:
{resume}

Job Description:
{job_desc}

Return matched and missing skills.
"""
)

score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
Give score (0-100) based on match.
Also explain why.

{match_data}
"""
)

In [ ]:
#Core Logic
def match_logic(resume, job_desc):
    resume_skills = resume.lower().split(", ")
    job_skills = job_desc.lower().split(", ")

    matched = [s for s in resume_skills if s in job_skills]
    missing = [s for s in job_skills if s not in resume_skills]

    score = int((len(matched) / len(job_skills)) * 100)

    explanation = f"""
Matched Skills: {matched}
Missing Skills: {missing}
Score based on skill match.
"""

    return score, explanation

In [ ]:
#Pipeline
def pipeline(resume):
    score, explanation = match_logic(resume, job_desc)

    return f"""
Score: {score}/100
Explanation:
{explanation}
"""

chain = RunnableLambda(pipeline)

In [ ]:
#Input Data
job_desc = "python, machine learning, nlp, sql"

resume_strong = "python, machine learning, nlp, sql, 3 years experience"
resume_avg = "python, sql, 1 year experience"
resume_weak = "basic computer knowledge"

In [ ]:
#Display Function
print("STRONG:\n", chain.invoke(resume_strong))
print("AVERAGE:\n", chain.invoke(resume_avg))
print("WEAK:\n", chain.invoke(resume_weak))

# Extra runs for tracing proof
chain.invoke("extra test 1")
chain.invoke("extra test 2")

STRONG:
 
Score: 100/100
Explanation:

Matched Skills: ['python', 'machine learning', 'nlp', 'sql']
Missing Skills: []
Score based on skill match.


AVERAGE:
 
Score: 50/100
Explanation:

Matched Skills: ['python', 'sql']
Missing Skills: ['machine learning', 'nlp']
Score based on skill match.


WEAK:
 
Score: 0/100
Explanation:

Matched Skills: []
Missing Skills: ['python', 'machine learning', 'nlp', 'sql']
Score based on skill match.




"\nScore: 0/100\nExplanation:\n\nMatched Skills: []\nMissing Skills: ['python', 'machine learning', 'nlp', 'sql']\nScore based on skill match.\n\n"


##  FINAL REPORT



Project: AI Resume Screening System

Objective:
Evaluate resumes based on job description and assign score with explanation.

Approach:
Used LangChain PromptTemplate and a modular pipeline to compare
candidate skills with required skills.

Features:
- Skill-based scoring
- Explanation for each result
- Clean and reusable pipeline
- LangSmith tracing for execution monitoring

Results:
Strong → 100/100  
Average → 50/100  
Weak → 0/100  

Conclusion:
This project shows how structured pipelines can be built using LangChain
for real-world applications like resume screening.
